# Sapiens: Oberflächennormalen statt Verpixelung

Nachbau der Vorverarbeitung aus **Ponnambalam et al., *Privacy-Centric Seizure Detection Using Surface Normals, Pose and Segmentation Masks*** (AI 2025, [DOI 10.1007/978-981-95-4972-6_25](https://doi.org/10.1007/978-981-95-4972-6_25)):
Jedes Bild wird durch eine **Karte der Oberflächennormalen** ersetzt (pro Pixel die Richtung, in die die Oberfläche zeigt, als Farbe). Farbe, Hautton und Gesichtszüge verschwinden, Form und Bewegung bleiben.

Modell, Quellen und alle Messwerte: [`docs/SAPIENS_LOG.md`](../docs/SAPIENS_LOG.md).

| Schritt | wie im Paper | hier |
|---|---|---|
| Bildrate | 5 fps | 5 fps (jeder 5. Frame von 25 fps) |
| Ausschnitt | YOLOv8-m-Personenbox | **ganze Ansicht** (Kamera zeigt schon aufs Bett), Farbe und S/W getrennt |
| Modell | Sapiens-2B Normalen | Sapiens-2B Normalen (TorchScript, in bfloat16) |

**Ablauf nach einem Kernel-Neustart:** Abschnitte **0** und **1** ausführen, dann in **2** den gewünschten Lauf.
Vorher den Kernel von `SAM3.ipynb` beenden: Sapiens braucht bis zu 7,3 von 8 GB GPU-Speicher.

**Vor jedem Commit: alle Ausgaben löschen** (Patientendaten!).

## 0. Setup

In [ ]:
from pathlib import Path
import pickle
import time

import cv2
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd().parent          # Notebook liegt in notebooks/, also eins höher
MODEL_FILE   = PROJECT_ROOT / "models" / "sapiens" / "sapiens_2b_normal_render_people_epoch_70_torchscript.pt2"
INPUT_VIDEO  = PROJECT_ROOT / "data" / "raw" / "PAT_1" / "VID_76.AVI"
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim" / "PAT_1"     # Zwischenergebnisse (.pkl)
OUTPUT_DIR   = PROJECT_ROOT / "data" / "processed" / "PAT_1"   # fertige Videos
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda"
DTYPE  = torch.bfloat16

for p in [MODEL_FILE, INPUT_VIDEO]:
    assert p.exists(), f"Fehlt: {p}"

print("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Modell laden: TorchScript (float32, 8,7 GB) erst in den Arbeitsspeicher, dann auf bfloat16 (4,4 GB) und auf die GPU.
# Die fertige bfloat16-Datei von Meta lässt sich mit torch 2.14 nicht laden (siehe Log).
t0 = time.time()
model = torch.jit.load(str(MODEL_FILE), map_location="cpu")
model = model.to(DTYPE).to(DEVICE).eval()
print(f"geladen in {time.time() - t0:.1f} s, GPU-Speicher belegt: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 1. Ansichten und Normalen-Funktion

Gleicher Bildaufbau wie in `SAM3.ipynb`: **Farbansicht** links (961 × 1080) und **S/W-Fenster** oben rechts (919 × 517). Jede Ansicht geht einzeln durch Sapiens. Alles außerhalb der Ansichten wird im fertigen Video schwarz.

**Vorbereitung für Sapiens** (Vorgaben aus der offiziellen Demo [`lite/demo/vis_normal.py`](https://github.com/facebookresearch/sapiens/blob/main/lite/demo/vis_normal.py)):

1. **Auffüllen:** Sapiens erwartet ein Hochkant-Bild 1024 × 768 (Höhe × Breite, also 4 : 3 hochkant). Beide Ansichten sind breiter. Damit nichts gestaucht wird, kommen oben und unten schwarze Streifen dazu, bis das Seitenverhältnis stimmt.
2. **Skalieren** auf 1024 × 768, BGR → RGB, normieren (Mittelwert `[123.5, 116.5, 103.5]`, Streuung `[58.5, 57.0, 57.5]`).
3. **Modell:** liefert 3 Kanäle (x, y, z) in halber Auflösung (512 × 384).
4. **Zurück:** auf die aufgefüllte Größe hochskalieren, Streifen wieder abschneiden, jeden Pixelvektor auf Länge 1 bringen.
5. **Farbbild:** x, y, z von [-1, 1] auf [0, 255]. Reihenfolge umdrehen, weil OpenCV BGR speichert (wie in der Demo).

**Bewegungsmaß** (zur Kontrolle ohne Hinsehen, später evtl. nützlich für die Anfallserkennung): mittlerer Winkel in Grad zwischen den Normalen eines Frames und des vorherigen verarbeiteten Frames (0,2 s davor). 0 = nichts bewegt sich.

In [ ]:
# Bildaufbau VID_76.AVI (Werte in Pixeln: x0, y0, x1, y1), wie in SAM3.ipynb
SPLIT_X = 961
SW_VIEW = (991, 10, 1910, 527)
VIEWS = {
    "F":  (0, 0, SPLIT_X, 1080),   # Farbe: linke Hälfte
    "SW": SW_VIEW,                 # S/W-Fenster, 10 px eingerückt
}

IN_H, IN_W = 1024, 768             # Eingabegröße Sapiens (Höhe, Breite)
MEAN = torch.tensor([123.5, 116.5, 103.5], device=DEVICE).view(1, 3, 1, 1)
STD  = torch.tensor([58.5, 57.0, 57.5], device=DEVICE).view(1, 3, 1, 1)


def pad_to_portrait(img):
    """Füllt oben/unten (oder links/rechts) schwarz auf, bis Breite : Höhe = 768 : 1024.
    Gibt das aufgefüllte Bild und die Lage des Originals darin zurück (top, left)."""
    h, w = img.shape[:2]
    target = IN_W / IN_H                       # 0,75
    if w / h > target:                         # zu breit -> oben/unten auffüllen
        new_h, new_w = int(round(w / target)), w
    else:                                      # zu schmal -> links/rechts auffüllen
        new_h, new_w = h, int(round(h * target))
    top, left = (new_h - h) // 2, (new_w - w) // 2
    padded = np.zeros((new_h, new_w, 3), dtype=img.dtype)
    padded[top:top + h, left:left + w] = img
    return padded, top, left


@torch.inference_mode()
def estimate_normals(view_bgr):
    """Oberflächennormalen einer Ansicht -> Tensor (3, H, W) auf der GPU, Länge 1 pro Pixel."""
    h, w = view_bgr.shape[:2]
    padded, top, left = pad_to_portrait(view_bgr)
    ph, pw = padded.shape[:2]
    rgb = cv2.cvtColor(cv2.resize(padded, (IN_W, IN_H), interpolation=cv2.INTER_LINEAR), cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(rgb).to(DEVICE).permute(2, 0, 1)[None].float()
    x = ((x - MEAN) / STD).to(DTYPE)

    out = model(x).float()                                                  # (1, 3, 512, 384)
    out = F.interpolate(out, size=(ph, pw), mode="bilinear", align_corners=False)[0]
    out = out[:, top:top + h, left:left + w]                                # Streifen abschneiden
    return out / (out.norm(dim=0, keepdim=True) + 1e-5)


def normals_to_bgr(n):
    """Normalen (3, H, W) -> Farbbild (H, W, 3) uint8 für OpenCV."""
    img = ((n + 1) / 2 * 255).clamp(0, 255).byte().permute(1, 2, 0).cpu().numpy()
    return np.ascontiguousarray(img[:, :, ::-1])


def angle_map_deg(a, b):
    """Winkel in Grad pro Pixel zwischen zwei Normalenkarten (3, H, W).
    atan2 statt acos: acos ist bei fast gleichen Richtungen ungenau (zeigt dann ~0,4° statt 0°)."""
    cross = torch.linalg.cross(a, b, dim=0).norm(dim=0)
    dot = (a * b).sum(dim=0)
    return torch.rad2deg(torch.atan2(cross, dot))


def mean_angle_deg(a, b):
    """Mittlerer Winkel in Grad zwischen zwei Normalenkarten."""
    return float(angle_map_deg(a, b).mean())


def run_normals(start=0, end=None, step=5):
    """Jeden `step`-ten Frame aus [start, end) in Normalen umwandeln. Speichert auch bei Abbruch."""
    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    end = total if end is None else min(end, total)
    for _ in range(start):          # bis zum Start vorlesen (bei AVI ist cap.set ungenau)
        cap.grab()

    stamp = time.strftime("%Y-%m-%d_%H-%M-%S")   # Datum + Uhrzeit -> nichts wird überschrieben
    stem = f"{INPUT_VIDEO.stem}_sapiens2b_{start}_{end}_{stamp}"
    out_fps = fps / step
    print(f"Dateien: {stem}  |  {fps:.0f} fps -> {out_fps:.0f} fps")
    writer = cv2.VideoWriter(str(OUTPUT_DIR / f"{stem}_normalen.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (w, h))

    per_frame, prev = [], {}
    n_out = len(range(start, end, step))
    t0 = time.time()
    torch.cuda.reset_peak_memory_stats()
    try:
        for k, frame_nr in enumerate(range(start, end, step)):
            ok, frame_bgr = cap.read()
            if not ok:
                break
            for _ in range(step - 1):            # übersprungene Frames nur weiterspulen
                cap.grab()

            out = np.zeros_like(frame_bgr)       # außerhalb der Ansichten: schwarz
            rec = {"frame": frame_nr}
            for v, (x0, y0, x1, y1) in VIEWS.items():
                t = time.time()
                n = estimate_normals(frame_bgr[y0:y1, x0:x1])
                out[y0:y1, x0:x1] = normals_to_bgr(n)
                rec[f"bewegung_{v}"] = mean_angle_deg(n, prev[v]) if v in prev else np.nan
                rec[f"zeit_{v}"] = time.time() - t
                prev[v] = n
            writer.write(out)
            per_frame.append(rec)

            if k % 25 == 0:                       # alle 25 Ausgabe-Frames = 5 s Video
                el = time.time() - t0
                print(f"Frame {frame_nr:5d} ({k}/{n_out})  Bewegung F {rec['bewegung_F']:5.2f}°  SW {rec['bewegung_SW']:5.2f}°  "
                      f"{el:7.1f} s  GPU max {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    finally:
        # Läuft immer, auch beim Stopp-Knopf -> Video abspielbar, Ergebnisse gespeichert
        cap.release()
        writer.release()
        el = time.time() - t0
        print(f"Ende: {len(per_frame)} Frames in {el:.1f} s ({el / max(len(per_frame), 1):.2f} s pro Frame, beide Ansichten)")
        with open(INTERIM_DIR / f"{stem}.pkl", "wb") as f:
            pickle.dump(per_frame, f)
    return per_frame


def summary(per_frame):
    """Tempo und Bewegungsmaß pro Ansicht."""
    for v, label in [("F", "Farbe"), ("SW", "S/W")]:
        z = np.array([r[f"zeit_{v}"] for r in per_frame])
        b = np.array([r[f"bewegung_{v}"] for r in per_frame])
        b = b[~np.isnan(b)]
        print(f"{label:5s}: {z.mean():.2f} s pro Bild | Bewegung (Grad) Median {np.median(b):.2f}, "
              f"95 % {np.percentile(b, 95):.2f}, max {b.max():.2f}")

## 2. Läufe

Ergebnisse: `data/processed/PAT_1/..._normalen.mp4` (5 fps, gleiches Bildformat wie die Pixel-Videos) und `data/interim/PAT_1/....pkl` (Bewegungsmaß und Zeit pro Frame).

Erst den kurzen Test laufen lassen und das Video anschauen.

In [ ]:
# Kurzer Test: 2 s Video (10 Ausgabe-Frames), ca. 1 min
res_test = run_normals(start=2400, end=2450)
summary(res_test)

In [ ]:
# Schwierigste Stelle aus SAM3 (Frames 2400–3000): 120 Ausgabe-Frames, ca. 9 min
res_2400 = run_normals(start=2400, end=3000)
summary(res_2400)

In [ ]:
# Ganzes Video: 8759 Frames -> 1752 Ausgabe-Frames, bei ~4,4 s pro Frame ca. 2 h 10 min
# (bricht man ab, werden Video und .pkl trotzdem sauber gespeichert)
res_full = run_normals()
summary(res_full)

---
## Anhang: Verfälscht bfloat16 die Normalen? (nicht im normalen Ablauf)

Das Modell läuft hier in **bfloat16** (weniger genaue Zahlen), weil float32 nicht auf die GPU passt. Dieser Test rechnet **einen** Frame zusätzlich mit dem Original in float32 auf der **CPU** und vergleicht die Winkel.
Zum Vergleich: Sapiens-2B hat laut Paper einen mittleren Winkelfehler von 11,84° gegenüber der Wahrheit. Liegt der Unterschied bfloat16/float32 deutlich darunter, ist bfloat16 unproblematisch.

**Achtung:** braucht zusätzlich ~9 GB Arbeitsspeicher (der Rechner hat 16 GB) und einige Minuten. Vorher andere Programme schließen.

In [ ]:
FRAME_NR, VIEW = 2400, "F"

cap = cv2.VideoCapture(str(INPUT_VIDEO))
for _ in range(FRAME_NR):
    cap.grab()
ok, frame_bgr = cap.read()
cap.release()
x0, y0, x1, y1 = VIEWS[VIEW]
view = frame_bgr[y0:y1, x0:x1]

n_bf16 = estimate_normals(view).cpu()

# Gleiche Vorverarbeitung, aber float32 auf der CPU
model32 = torch.jit.load(str(MODEL_FILE), map_location="cpu").eval()
h, w = view.shape[:2]
padded, top, left = pad_to_portrait(view)
rgb = cv2.cvtColor(cv2.resize(padded, (IN_W, IN_H)), cv2.COLOR_BGR2RGB)
x = torch.from_numpy(rgb).permute(2, 0, 1)[None].float()
x = (x - MEAN.cpu()) / STD.cpu()
t0 = time.time()
with torch.inference_mode():
    out = model32(x).float()
print(f"float32 auf CPU: {time.time() - t0:.0f} s")
out = F.interpolate(out, size=padded.shape[:2], mode="bilinear", align_corners=False)[0][:, top:top + h, left:left + w]
n_fp32 = out / (out.norm(dim=0, keepdim=True) + 1e-5)

ang = angle_map_deg(n_bf16, n_fp32)
print(f"Winkel bfloat16 vs. float32 (Grad): Mittel {ang.mean():.2f}, Median {ang.median():.2f}, "
      f"95 % {ang.flatten().quantile(0.95):.2f}, max {ang.max():.2f}")

del model32

## 3. Notizen

Ergebnisse und Entscheidungen werden in [`docs/SAPIENS_LOG.md`](../docs/SAPIENS_LOG.md) festgehalten.